In [16]:
ADLSaccount = "fabmigration"
containerName = "fabmigration"
sourceFolder = "bronze/adventureworks2"
targetFolder = "silver/adventureworks"
targetEnv = "silver"
entityName = "Sales.CurrencyRate"
fileExtension = "parquet"

In [17]:
%run /RetailSilver/00_Functions

In [18]:
# Set file path to load to DF
sourcePath = f'abfss://{containerName}@{ADLSaccount}.dfs.core.windows.net/{sourceFolder}/{entityName}.{fileExtension}'
targetPath = f'abfss://{containerName}@{ADLSaccount}.dfs.core.windows.net/{targetFolder}/'

In [19]:
# Read parquet file to get data
dfdata = spark.read.load(sourcePath, format='parquet')

# Clean
dfdata = clean_df(dfdata)

# Set the schema for entity. Also if needed the first row.
entitySchema = GetSaleTableSchema(entityName)
firstRow = GetSaleTableFirstRow(entityName, dfdata)

# Create empty DF with schema and first row (if needed)
df = spark.createDataFrame(firstRow, entitySchema)

# Update column names from parquet file if needed
dfdata = UpdateDFColNames(entityName, dfdata)

# Join empty Df with data from parquet
df = df.unionByName(dfdata)

#display(df.limit(10))

In [20]:
# Set name for silver
entityNameFix = f'silver_{entityName.replace(".", "_")}'

# Save DF into stage after clean process
SaveDF(df, targetEnv, entityNameFix, targetPath)